# Governance-Ready Fraud Decisioning: End-to-End Reproduction Notebook

**Purpose.** This notebook provides a single, commented orchestration layer for the MSc dissertation pipeline: baseline fraud modelling, Evidence Object generation, template narratives, constrained LLM narratives, audit summaries, stability checks, drift diagnostics, thin-file robustness evidence, and portability appendix artefacts.

**Important operating principle.** Heavy steps are parameterised and guarded. The notebook is designed to be run in GitHub Codespaces from the repository root. Long-running LLM steps should be run resume-safe and monitored rather than re-run unnecessarily.

**Final thesis alignment.** The notebook is aligned to the proposal commitments and final dissertation evidence: original 5,753-row LLM run retained as intermediate evidence, final 20,000-row LLM robustness run used as the full-scale LLM result, thin-file re-score attempted closure documented, and portability treated as protocol-level transfer only.

## 0. Environment and execution switches

Set execution switches carefully. Defaults avoid re-running expensive work. Turn switches on only when rebuilding artefacts intentionally.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

REPO = Path.cwd()
ART = REPO / "artifacts" / "baselines" / "lgbm_numeric_v1_subsample"
SPLIT = REPO / "artifacts" / "splits" / "v1_temporal_q70_q85"
DATA = REPO / "data" / "ieee-cis"

os.environ.setdefault("PYTHONPATH", "src")
os.environ.setdefault("IEEE_CIS_DIR", str(DATA))

RUN_HEAVY_BASELINE = False
RUN_TEMPLATE_PIPELINE = False
RUN_LLM_20000 = False
RUN_STABILITY_REGEN = False
RUN_DRIFT_SUITE = False
RUN_SUMMARIES_ONLY = True

print("Repo:", REPO)
print("Artifact dir:", ART)
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))
print("IEEE_CIS_DIR:", os.environ.get("IEEE_CIS_DIR"))

## 1. Proposal-to-thesis traceability checkpoint

This cell records the final evidence position. It is deliberately concise and examiner-safe.

In [ ]:
traceability = [
    ("IEEE-CIS baseline", "LightGBM baseline; ROC-AUC 0.8687; PR-AUC 0.4594; Brier 0.0236; ECE 0.0043", "Complete"),
    ("Evidence Object schema", "20,000 enriched EOs plus TransactionDT-enriched artefact", "Complete"),
    ("Template narratives", "20,000 deterministic narratives with complete evidence rendering", "Complete"),
    ("Constrained LLM narratives", "Original 5,753 run retained; final 20,000-row audited robustness run", "Complete"),
    ("Validator taxonomy", "20,000 audit records; 7,994 clean; 12,006 driver-omission flagged; 0 fallbacks", "Substantially complete"),
    ("Stability/randomisation", "800 regenerated outputs across original/shuffled/perturbed/randomised variants", "Materially strengthened"),
    ("Drift awareness", "TransactionDT windows plus PSI/KS/Wasserstein/TVD/JSD diagnostics", "Offline diagnostic support"),
    ("Thin-file robustness", "Disclosure evidence and EO masking proxy; feature re-score attempted, terminated by environment", "Partially supported"),
    ("Human simulatability", "Automated proxy fallback used", "Future work"),
    ("Portability", "Protocol-level fraud/cyber/crypto mapping appendix", "Optional item addressed"),
]
for row in traceability:
    print(row)

## 2. Repository hygiene preflight

Run this before committing. It records status, key files, and obvious large artefacts. Use the output to decide what should be committed versus kept as untracked/generated artefacts.

In [ ]:
def sh(cmd, check=False):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, text=True, capture_output=False)

sh("git branch --show-current")
sh("git status --short")
sh("find scripts -maxdepth 1 -type f | sort | sed -n '1,200p'")
sh("find artifacts/baselines/lgbm_numeric_v1_subsample -maxdepth 2 -type f -printf '%s %p\n' 2>/dev/null | sort -nr | head -n 40")

## 3. Baseline and EO pipeline

Use these cells only if rebuilding the baseline. For submission hygiene, prefer using already validated artefacts unless a deliberate rebuild is required.

In [ ]:
if RUN_HEAVY_BASELINE:
    sh("python scripts/make_split_v1.py", check=True)
    sh("python scripts/materialize_joined_train.py", check=True)
    sh("python scripts/train_lgbm_numeric_v1.py", check=True)
    sh("python scripts/emit_eos_v1.py", check=True)
    sh("python scripts/attach_shap_drivers_v1.py", check=True)
else:
    print("Skipping heavy baseline rebuild. Set RUN_HEAVY_BASELINE=True to run.")

## 4. Template and LLM narrative generation

Template generation is comparatively cheap. The full 20,000-row LLM robustness run is expensive and should only be relaunched if outputs are incomplete. It is resume-safe.

In [ ]:
if RUN_TEMPLATE_PIPELINE:
    sh("python scripts/orchestrate_narrative_experiments.py", check=True)
    sh("python scripts/orchestrate_evaluation.py", check=True)
else:
    print("Skipping template pipeline. Set RUN_TEMPLATE_PIPELINE=True to run.")

if RUN_LLM_20000:
    robust = ART / "llm_20000_robustness"
    robust.mkdir(parents=True, exist_ok=True)
    cmd = "python scripts/run_llm_20000_resume_safe.py --max-rows 20000 --max-retries 2"
    sh(cmd, check=True)
else:
    print("Skipping full LLM 20,000 run. Set RUN_LLM_20000=True only if needed.")

## 5. Summarise final LLM robustness evidence

This should be safe to run after the 20,000-row output and audit JSONL exist.

In [ ]:
summary_script = REPO / "scripts" / "summarise_llm_20000_robustness.py"
if summary_script.exists():
    sh("python scripts/summarise_llm_20000_robustness.py", check=True)
    sh("cat artifacts/baselines/lgbm_numeric_v1_subsample/llm_20000_robustness/llm_20000_robustness_summary.md", check=False)
else:
    print("Missing summarise_llm_20000_robustness.py. Recreate from project notes if needed.")

## 6. Stability, drift, thin-file and portability artefacts

Run summaries/diagnostics as needed. Stability regeneration and LLM variants are expensive; drift summaries are cheap if EO/prediction artefacts exist.

In [ ]:
if RUN_STABILITY_REGEN:
    sh("python scripts/create_regeneration_stability_variants.py", check=True)
    print("Then run LLM narratives for each variant if required. See runbook appendix.")
else:
    print("Skipping stability regeneration. Existing summary can be inspected below.")

if (ART / "regeneration_stability" / "regeneration_stability_summary.md").exists():
    sh("cat artifacts/baselines/lgbm_numeric_v1_subsample/regeneration_stability/regeneration_stability_summary.md")

if RUN_DRIFT_SUITE:
    sh("python scripts/evaluate_production_style_drift_metrics.py", check=True)

if (ART / "drift_metric_suite" / "drift_metric_suite_summary.md").exists():
    sh("cat artifacts/baselines/lgbm_numeric_v1_subsample/drift_metric_suite/drift_metric_suite_summary.md")

for path in [
    ART / "feature_masking_rescore" / "thin_file_re_score_attempt_closure.md",
    ART / "portability_appendix" / "eo_protocol_portability_appendix.md",
]:
    if path.exists():
        print("
---", path, "---")
        print(path.read_text(encoding="utf-8")[:4000])

## 7. Final artefact manifest

This manifest is intended to support submission/review. It documents the most important artefacts without requiring large generated files to be committed to Git.

In [ ]:
manifest = {
    "baseline": {
        "metrics": str(ART / "metrics.json"),
        "predictions": str(ART / "test_predictions.csv"),
        "model": str(ART / "model.txt"),
    },
    "evidence_objects": {
        "base": str(ART / "eos_test.jsonl"),
        "with_drivers": str(ART / "eos_test_with_drivers.jsonl"),
        "with_transactiondt": str(ART / "eos_test_with_drivers_with_transactiondt.jsonl"),
    },
    "narratives": {
        "template_20000": str(ART / "narratives_ops_triage_template.jsonl"),
        "llm_original_5753": str(ART / "narratives_ops_triage_llm_5753rows_backup.jsonl"),
        "llm_robustness_20000": str(ART / "llm_20000_robustness" / "narratives_ops_triage_llm_20000_resume_safe.jsonl"),
    },
    "audit": {
        "llm_20000_attempt_audit": str(ART / "llm_20000_robustness" / "llm_20000_attempt_audit.jsonl"),
        "llm_20000_summary": str(ART / "llm_20000_robustness" / "llm_20000_robustness_summary.md"),
    },
    "proposal_gap_uplifts": {
        "drift_suite": str(ART / "drift_metric_suite" / "drift_metric_suite_summary.md"),
        "stability_summary": str(ART / "regeneration_stability" / "regeneration_stability_summary.md"),
        "thin_file_attempt_closure": str(ART / "feature_masking_rescore" / "thin_file_re_score_attempt_closure.md"),
        "portability_appendix": str(ART / "portability_appendix" / "eo_protocol_portability_appendix.md"),
    },
}
manifest_path = ART / "final_submission_artifact_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "
", encoding="utf-8")
print(manifest_path)
print(json.dumps(manifest, indent=2))

## 8. Final hygiene checklist before commit

Recommended commit surface:

- Track: source scripts, notebooks, README/runbook, summary `.md/.json/.csv` artefacts that are small.
- Do not track: raw data, large JSONL outputs, model binaries if repository policy excludes generated artefacts.
- Archive externally if required: full 20,000 LLM JSONL output and audit JSONL.

In [ ]:
sh("git status --short")
print("
Suggested next shell steps:")
print("git add scripts notebooks README.md docs artifacts/baselines/lgbm_numeric_v1_subsample/*summary*.md")
print("git commit -m 'final thesis reproducibility and governance artefacts'")
print("git tag thesis-final-v12-20260714")